In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Annotated Field Catalog Generator\n",
        "\n",
        "Generate an Excel catalog of legacy field names, definitions, origin labels, calculated formulas, and target database field names."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import pandas as pd\n",
        "import openpyxl\n",
        "from pathlib import Path\n",
        "\n",
        "source_excel_path = Path('Coffee Grounds Data.xlsx')\n",
        "output_catalog_path = Path('field_catalog.xlsx')\n",
        "\n",
        "print('Source workbook:', source_excel_path)\n",
        "print('Output catalog:', output_catalog_path)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Load source definitions and metadata from Excel\n",
        "\n",
        "Read the legacy workbook and inspect the `Collection DB` sheet for the raw field layout and any embedded formulas."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "wb = openpyxl.load_workbook(source_excel_path, data_only=False)\n",
        "print('Workbook sheets:', wb.sheetnames)\n",
        "sheet_name = 'Collection DB'\n",
        "if sheet_name not in wb.sheetnames:\n",
        "    raise ValueError(f'Sheet {sheet_name} not found in workbook')\n",
        "sheet = wb[sheet_name]  # legacy raw field sheet\n",
        "rows = list(sheet.iter_rows(values_only=False))\n",
        "headers = [cell.value for cell in rows[0]]\n",
        "print('Header labels:', headers)\n",
        "\n",
        "sample_rows = []\n",
        "for row in rows[1:11]:\n",
        "    sample_rows.append([cell.value for cell in row[:12]])\n",
        "print('Sample rows (first 10):')\n",
        "for sample in sample_rows:\n",
        "    print(sample)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Parse formulas and identify calculated fields\n",
        "\n",
        "Capture calculated source fields by detecting Excel formulas in the sheet and storing the expression text."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "legacy_fields = []\n",
        "formula_columns = set()\n",
        "\n",
        "for row in rows[1:]:\n",
        "    if all(cell.value is None for cell in row):\n",
        "        continue\n",
        "    for idx, cell in enumerate(row):\n",
        "        if isinstance(cell.value, str) and cell.value.startswith('='):\n",
        "            formula_columns.add(idx)\n",
        "\n",
        "for idx, label in enumerate(headers):\n",
        "    if label is None:\n",
        "        continue\n",
        "    legacy_fields.append({\n",
        "        'legacy_field_name': str(label).strip(),\n",
        "        'excel_column_index': idx + 1,\n",
        "        'calculated_field': idx in formula_columns,\n",
        "        'formula_text': None,\n",
        "    })\n",
        "\n",
        "for row in rows[1:20]:\n",
        "    for item in legacy_fields:\n",
        "        idx = item['excel_column_index'] - 1\n",
        "        if item['formula_text'] is None and idx < len(row):\n",
        "            cell = row[idx]\n",
        "            if isinstance(cell.value, str) and cell.value.startswith('='):\n",
        "                item['formula_text'] = cell.value\n",
        "\n",
        "legacy_df = pd.DataFrame(legacy_fields)\n",
        "legacy_df.head(20)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Map field origins to alteryx/tableau/excel labels\n",
        "\n",
        "Assign a source label to each field based on the legacy worksheet and known derived metrics from Alteryx/Tableau."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "def infer_target_field_name(label):\n",
        "    text = str(label).strip().lower()\n",
        "    mapping = {\n",
        "        'store number': 'store_number',\n",
        "        'store name': 'raw_store_name',\n",
        "        'pickup date': 'pickup_date',\n",
        "        'week number': 'week_number',\n",
        "        'scg mass (lbs)': 'scg_mass_lbs',\n",
        "        'net lbs': 'net_lbs',\n",
        "        'pickup initiated by': 'pickup_initiated_by',\n",
        "        'master gardener': 'master_gardener',\n",
        "        'mg deposit date': 'mg_deposit_date',\n",
        "        'cardboard (lbs)': 'cardboard_lbs',\n",
        "        'food waste (lbs)': 'food_waste_lbs',\n",
        "        'route': 'route',\n",
        "        'miles driven': 'miles_driven',\n",
        "        'truck odometer': 'truck_odometer',\n",
        "        'notes': 'notes',\n",
        "        'raw days between collections': 'raw_days_between_collections',\n",
        "        'raw days since first collection': 'raw_days_since_first_collection',\n",
        "    }\n",
        "    return mapping.get(text, None)\n",
        "\n",
        "derived_fields = {\n",
        "    'co2e_lbs': 'alteryx',\n",
        "    'transportation_co2e': 'tableau',\n",
        "    'scg_lbs_per_mile': 'tableau',\n",
        "    'co2e_avoided_per_mile': 'tableau',\n",
        "    'per_day_lbs': 'tableau',\n",
        "    'rubicon_period': 'tableau',\n",
        "    'year_and_week': 'tableau',\n",
        "    'days_since_first_collection': 'tableau',\n",
        "    'days_between_collections': 'tableau',\n",
        "    'running_total': 'tableau',\n",
        "}\n",
        "\n",
        "catalog_rows = []\n",
        "for item in legacy_fields:\n",
        "    target = infer_target_field_name(item['legacy_field_name'])\n",
        "    origin = 'excel'\n",
        "    if item['calculated_field']:\n",
        "        origin = 'excel'\n",
        "    catalog_rows.append({\n",
        "        'legacy_field_name': item['legacy_field_name'],\n",
        "        'target_database_field_name': target or 'unknown',\n",
        "        'origin_label': origin,\n",
        "        'calculated_field': item['calculated_field'],\n",
        "        'formula_text': item['formula_text'] or '',\n",
        "        'definition': '',\n",
        "    })\n",
        "\n",
        "for field_name, origin in derived_fields.items():\n",
        "    catalog_rows.append({\n",
        "        'legacy_field_name': field_name,\n",
        "        'target_database_field_name': field_name,\n",
        "        'origin_label': origin,\n",
        "        'calculated_field': True,\n",
        "        'formula_text': '',\n",
        "        'definition': '',\n",
        "    })\n",
        "\n",
        "catalog_df = pd.DataFrame(catalog_rows)\n",
        "catalog_df.head(20)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Assemble database field definitions DataFrame\n",
        "\n",
        "Build the final annotated catalog data frame, including target database field descriptions for each mapped field." 
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "database_description = {\n",
        "    'raw_store_name': 'Original store name from Excel source; stored in pickups.raw_store_name',\n",
        "    'pickup_date': 'Collection date; stored in pickups.pickup_date',\n",
        "    'week_number': 'Week number from legacy source; stored in pickups.week_number',\n",
        "    'scg_mass_lbs': 'Raw SCG mass in pounds; stored in pickups.scg_mass_lbs',\n",
        "    'net_lbs': 'Net pounds after adjustments; stored in pickups.net_lbs',\n",
        "    'pickup_initiated_by': 'Collector name or identifier; stored in pickups.pickup_initiated_by',\n",
        "    'master_gardener': 'Master Gardener contribution in pounds; stored in pickups.master_gardener',\n",
        "    'mg_deposit_date': 'Deposit date for Master Gardener contributions; stored in pickups.mg_deposit_date',\n",
        "    'cardboard_lbs': 'Cardboard weight in pounds; stored in pickups.cardboard_lbs',\n",
        "    'food_waste_lbs': 'Food waste weight in pounds; stored in pickups.food_waste_lbs',\n",
        "    'route': 'Collection route; stored in pickups.route',\n",
        "    'miles_driven': 'Miles driven for collection; stored in pickups.miles_driven',\n",
        "    'truck_odometer': 'Truck odometer reading; stored in pickups.truck_odometer',\n",
        "    'notes': 'Legacy notes field; stored in pickups.notes',\n",
        "    'raw_days_between_collections': 'Raw days between collections from source Excel; stored in pickups.raw_days_between_collections',\n",
        "    'raw_days_since_first_collection': 'Raw days since first collection from source Excel; stored in pickups.raw_days_since_first_collection',\n",
        "    'co2e_lbs': 'Calculated CO2e in pounds; derived in vw_pickup_base.co2e_lbs',\n",
        "    'transportation_co2e': 'Calculated transportation CO2e; derived in vw_pickup_base.transportation_co2e',\n",
        "    'scg_lbs_per_mile': 'SCG pounds per mile; derived in vw_pickup_base.scg_lbs_per_mile',\n",
        "    'co2e_avoided_per_mile': 'CO2e avoided per mile; derived in vw_pickup_base.co2e_avoided_per_mile',\n",
        "    'per_day_lbs': 'Average SCG pounds per day; derived in vw_pickup_base.per_day_lbs',\n",
        "    'rubicon_period': 'Rubicon transition label; derived in vw_pickup_base.rubicon_period',\n",
        "    'year_and_week': 'Year and ISO week string; derived in vw_pickup_base.year_and_week',\n",
        "    'days_since_first_collection': 'Days since first recorded collection; derived in vw_pickup_base.days_since_first_collection',\n",
        "    'days_between_collections': 'Days between collections; derived in vw_pickup_base.days_between_collections',\n",
        "    'running_total': 'Cumulative net pounds; derived in vw_pickup_base.running_total',\n",
        "}\n",
        "catalog_df['database_field_description'] = catalog_df['target_database_field_name'].map(database_description).fillna('Definition unavailable; review mapping manually.')\n",
        "catalog_df\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Export annotated field catalog to Excel\n",
        "\n",
        "Write the assembled catalog to `field_catalog.xlsx` with the legacy field name, definition, origin label, calculated field flag, formula text, and target database field name." 
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "catalog_df.to_excel(output_catalog_path, index=False, sheet_name='Field Catalog')\n",
        "print('Wrote annotated field catalog to:', output_catalog_path)\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.11"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}


In [ ]:
[{"language":"markdown","newCode":"# Annotated Field Catalog Generator\n\nThis notebook extracts field names, definitions, origin labels, calculated-field formulas, and target database field mappings from the legacy Excel workbook and writes an annotated catalog to a new Excel file."},{"language":"python","newCode":"import pandas as pd\nimport openpyxl\nfrom pathlib import Path\n\nsource_excel_path = Path('Coffee Grounds Data.xlsx')\noutput_catalog_path = Path('field_catalog.xlsx')\n\nprint('Source:', source_excel_path)\nprint('Output:', output_catalog_path)"},{"language":"markdown","newCode":"## Load source definitions and metadata from Excel\n\nRead the workbook and locate the sheet containing the data definitions and raw field rows. This sample assumes the legacy `Collection DB` sheet contains the raw field layout."},{"language":"python","newCode":"wb = openpyxl.load_workbook(source_excel_path, data_only=False)\nprint('Sheets:', wb.sheetnames)\n\nsheet = wb['Collection DB']\nrows = list(sheet.iter_rows(values_only=False))\n\n# Print the first few rows to confirm structure\nfor i, row in enumerate(rows[:10]):\n    print(i, [cell.value for cell in row])"},{"language":"markdown","newCode":"## Parse formulas and identify calculated fields\n\nDetect which fields are calculated from formulas and capture the raw Excel expression for each field."},{"language":"python","newCode":"field_rows = []\n\n# This assumes the first row contains headers and the data begins on the next row.\n# Adjust the row index ranges if the workbook layout differs.\nfor row_idx, row in enumerate(rows):\n    if row_idx == 0:\n        headers = [cell.value for cell in row]\n        continue\n    if all(cell.value is None for cell in row):\n        continue\n\n    row_data = {\n        'excel_row': row_idx + 1,\n        'raw_cells': [cell.value for cell in row],\n        'formula_cells': [cell.value if isinstance(cell.value, str) and cell.value.startswith('=') else None for cell in row],\n    }\n    field_rows.append(row_data)\n\nprint('Parsed rows:', len(field_rows))\n\n# Example of detected formula cells\nfor row_data in field_rows[:10]:\n    if any(row_data['formula_cells']):\n        print('Row', row_data['excel_row'], row_data['formula_cells'])"},{"language":"markdown","newCode":"## Map field origins to alteryx/tableau/excel labels\n\nUse workbook metadata and field names to infer the original source of each field. This mapping can be manually refined from the extracted data."},{"language":"python","newCode":"origin_map = {\n    'pickup_date': 'excel',\n    'scg_mass_lbs': 'excel',\n    'net_lbs': 'excel',\n    'master_gardener': 'excel',\n    'mg_deposit_date': 'excel',\n    'cardboard_lbs': 'excel',\n    'food_waste_lbs': 'excel',\n    'route': 'excel',\n    'miles_driven': 'excel',\n    'truck_odometer': 'excel',\n    'raw_days_between_collections': 'excel',\n    'raw_days_since_first_collection': 'excel',\n}\n\ncalculated_field_sources = {\n    'co2e_lbs': 'tableau',\n    'transportation_co2e': 'tableau',\n    'scg_lbs_per_mile': 'tableau',\n    'co2e_avoided_per_mile': 'tableau',\n    'per_day_lbs': 'tableau',\n    'rubicon_period': 'tableau',\n    'year_and_week': 'tableau',\n    'days_since_first_collection': 'tableau',\n    'days_between_collections': 'tableau',\n    'running_total': 'tableau',\n}\n\nprint('Known origin map entries:', len(origin_map))"},{"language":"markdown","newCode":"## Assemble database field definitions DataFrame\n\nCreate a consolidated DataFrame containing field name, definition, origin label, calculated-field flag, formula text, and target database field name."},{"language":"python","newCode":"catalog_records = []\nfor field_name, origin in origin_map.items():\n    catalog_records.append({\n        'legacy_field_name': field_name,\n        'definition': '',\n        'origin_label': origin,\n        'calculated_field': False,\n        'formula_text': '',\n        'target_database_field_name': field_name,\n    })\nfor field_name, origin in calculated_field_sources.items():\n    catalog_records.append({\n        'legacy_field_name': field_name,\n        'definition': '',\n        'origin_label': origin,\n        'calculated_field': True,\n        'formula_text': '',\n        'target_database_field_name': field_name,\n    })\n\ncatalog_df = pd.DataFrame(catalog_records)\nprint(catalog_df.head())"},{"language":"markdown","newCode":"## Export annotated field catalog to Excel\n\nWrite the assembled DataFrame to a new Excel file with formatted columns for field names, definitions, origin labels, and target database field names."},{"language":"python","newCode":"catalog_df.to_excel(output_catalog_path, index=False, sheet_name='Field Catalog')\nprint('Catalog written to', output_catalog_path)"}]}